## Evaluate Best Models of Each Configuration

to create Table 2

In [ ]:
import os
import json
import sys

import numpy as np
import pandas as pd
import xarray as xr

base_dir = os.path.dirname(os.path.abspath('')).split(os.sep + 'evaluations')[0]

sys.path.append(os.path.sep.join([base_dir , 'src']))
from help_fcts import get_rmse, get_mae, get_mbe, get_r2

MODE = 'test'  # use test set!

In [2]:
scaling_factor = 1.796553703424 / 58391    # scaling_factor from mm w.e. to Gt for Greenland (specific to HIRHAM5 domain)
data_dir = os.path.sep.join([base_dir, 'data', 'processed', 'HIRHAM5-ERAInterim', 'v_02'])

train_test_split_file = 'train_val_test_split_100.json'
train_val_test_indices = json.load(open(os.path.sep.join([data_dir, train_test_split_file])))
valtest_indices = train_val_test_indices[MODE]

### Load Climatology (1990-2013) as baseline

In [3]:
interim_dir = os.path.sep.join([base_dir, 'data', 'interim', 'ERAI', 'HIRHAM5', 'firnpack'])
clim_melt = xr.open_dataset(os.path.sep.join([interim_dir, 'climat_1990-2013_smoothed15.nc']))['snmel']
clim_melt_z = clim_melt.stack(z=('x', 'y')).reset_index('z').dropna('z', how='any')

In [4]:
# load base dataset
data_dir = os.path.sep.join([base_dir, 'data', 'processed', 'HIRHAM5-ERAInterim', 'v_02'])
ds = xr.open_zarr(os.path.sep.join([data_dir, f'base_dataset.zarr']), chunks='auto')
ds = ds.sel(time=valtest_indices)
ds_melt_z = ds['snmel']
ds_melt = ds_melt_z.set_index(z=['y', 'x']).unstack('z')

In [5]:
# Calculate daily anomalies of true validation data w.r.t. the climatology
md_clim = clim_melt_z['time'].dt.strftime('%m-%d')
clim_melt_z = clim_melt_z.assign_coords(md=md_clim)
clim_by_md = clim_melt_z.groupby('md').mean(dim='time')
md_data = ds_melt_z['time'].dt.strftime('%m-%d')
ds_melt_z = ds_melt_z.assign_coords(md=('time', md_data.values))
anomaly = clim_by_md - ds_melt_z.groupby('md')
anoms_val_z = anomaly.compute()
anoms_val = anoms_val_z.set_index(z=['y', 'x']).unstack('z')

### Performance of Climatology as Model

In [6]:
# climatological anomaly per location of valdiation
print(f'GrIS:')
gris_scores = {'rmse': get_rmse(anoms_val_z, 2),
               'mae': get_mae(anoms_val_z, 2),
               'mbe': get_mbe(anoms_val_z, 2),
               'R2': get_r2(ds_melt_z.values, clim_melt_z.where(clim_melt_z.md.isin(ds_melt_z.md), drop=True).values, 2)
}

print(f'RMSE: {gris_scores["rmse"]}')
print(f'MAE: {gris_scores["mae"]}')
print(f'MBE: {gris_scores["mbe"]}')
print(f'R2: {gris_scores["R2"]}\n')

GrIS:
RMSE: 2.3
MAE: 0.56
MBE: -0.17
R2: 0.78



### Predictions on test set for best model of each configuration

In [7]:
out_dir = os.path.sep.join([base_dir, 'output'])

best_models = [('regression', 0), ('shorttermNN', 28), ('modularNN', 21),  ('modularNN_noseason', None), 
               ('autoreg1noise', 18), ('modularNN_multitarget_trainable', 23), ('modularNN_albedo', None),]

df = pd.DataFrame()

for modelname, best_trial in best_models:
    if modelname == 'autoreg1noise':
        inf_values = ['', '_auto']
    else:
        inf_values = ['']
    for inf in inf_values:
        if best_trial is not None:
            model_spec = f'optuna_melt_{modelname}/trial_{best_trial}'
        else:
            # for model runs without hyperparameter optimization: modularNN_albedo, modularNN_noseason
            model_spec = f'optuna_melt_{modelname}'

        model_dir = os.path.sep.join([out_dir, f'{model_spec}/pred{inf}_{MODE}.zarr'])
        print(f'Load model: {model_dir}')
        ds_model = xr.open_dataset(model_dir)
        ds_model = ds_model.stack(z=('x', 'y')).reset_index('z')
        nan_mask = ~ds_model['snmel_true'].isnull().all(dim=["time"]).compute()
        ds_model = ds_model.isel(z=nan_mask)

        if 'true' not in df.columns:        
            stacked = ds_model['snmel_true'].stack(flat=("time", "z"))
            df = stacked.to_dataframe(name="true").reset_index(drop=True)   # columns: time, z, value
        
        df[modelname+inf] = ds_model['snmel_pred'].stack(flat=("time", "z")).values

df.head()


Load model: /home/eschlager/GRAMMLET/output/optuna_melt_regression/trial_0/pred_test.zarr
Load model: /home/eschlager/GRAMMLET/output/optuna_melt_shorttermNN/trial_28/pred_test.zarr
Load model: /home/eschlager/GRAMMLET/output/optuna_melt_modularNN/trial_21/pred_test.zarr
Load model: /home/eschlager/GRAMMLET/output/optuna_melt_modularNN_noseason/pred_test.zarr
Load model: /home/eschlager/GRAMMLET/output/optuna_melt_autoreg1noise/trial_18/pred_test.zarr
Load model: /home/eschlager/GRAMMLET/output/optuna_melt_autoreg1noise/trial_18/pred_auto_test.zarr
Load model: /home/eschlager/GRAMMLET/output/optuna_melt_modularNN_multitarget_trainable/trial_23/pred_test.zarr
Load model: /home/eschlager/GRAMMLET/output/optuna_melt_modularNN_albedo/pred_test.zarr


,lon,lat,x,y,time,z,true,regression,shorttermNN,modularNN,modularNN_noseason,autoreg1noise,autoreg1noise_auto,modularNN_multitarget_trainable,modularNN_albedo
0,-49.990738,62.442036,51,157,2016-01-01 12:00:00,0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.125390,0.0
1,-50.042488,62.485939,51,158,2016-01-01 12:00:00,1,0.0,0.0,0.0,0.0,0.0,0.005569,0.000326,0.277616,0.0
2,-50.094395,62.529823,51,159,2016-01-01 12:00:00,2,0.0,0.0,0.0,0.0,0.0,0.015662,0.013102,0.209262,0.0
3,-50.146450,62.573692,51,160,2016-01-01 12:00:00,3,0.0,0.0,0.0,0.0,0.0,0.014380,0.012568,0.097952,0.0
4,-49.896290,62.465771,52,157,2016-01-01 12:00:00,4,0.0,0.0,0.0,0.0,0.0,0.009441,0.000000,0.003384,0.0


In [8]:
# Add climatology to the dataframe
points = np.arange(len(df))
x_da = xr.DataArray(df['x'].values, dims=('points',), coords={'points': points})
y_da = xr.DataArray(df['y'].values, dims=('points',), coords={'points': points})
t_da = xr.DataArray(df['time'].values, dims=('points',), coords={'points': points})

clim_by_md = clim_by_md.set_index(z=['y', 'x']).unstack('z')
sampled = clim_by_md.sel(x=x_da, y=y_da, md=t_da.dt.strftime('%m-%d'))
df['climatology'] = sampled.values

In [9]:
# Table 2: Evaluation scores for best models on test set
df_scores = pd.DataFrame(columns=['rmse', 'mae', 'mbe', 'R2', 'R2_anomaly'])
modelnames = [x[0] for x in best_models]

for name in df.columns[7:]:
    residual = df[name] - df['true']
    rmse = get_rmse(residual, 2)
    mae = get_mae(residual, 2)
    mbe = get_mbe(residual, 2)
    r2 = get_r2(df['true'].values, df[name].values, 2)
    r2_anomaly = get_r2(df['true'].values-df['climatology'].values, df[name].values-df['climatology'].values, 2)
    df_scores.loc[name] = [rmse, mae, mbe, r2, r2_anomaly]

df_scores

,rmse,mae,mbe,R2,R2_anomaly
regression,1.60,0.40,0.05,0.89,0.51
shorttermNN,1.23,0.26,0.01,0.94,0.71
modularNN,0.90,0.18,-0.01,0.97,0.85
modularNN_noseason,0.96,0.19,-0.02,0.96,0.83
autoreg1noise,0.40,0.08,0.01,0.99,0.97
autoreg1noise_auto,0.90,0.15,-0.00,0.97,0.85
modularNN_multitarget_trainable,0.90,0.17,-0.01,0.97,0.85
modularNN_albedo,0.24,0.05,-0.00,1.00,0.99
climatology,2.30,0.56,-0.17,0.78,-0.01
